##### Inporting Lybraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime 

##### Importing Dataset

In [51]:
# Importing data and dropping null values

df = pd.read_excel('Online%20Retail.xlsx')
df = df.dropna(subset=['CustomerID', 'Description'])
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


#### Product Performance Matrix

##### Calculating Product Matrics

In [55]:
# creating new column called 'Total_revenue'

df['Total_revenue'] = df['Quantity'] * df['UnitPrice']

In [50]:
product_performance = df.groupby(['StockCode', 'Description']).agg({
    'Total_revenue': 'sum',
    'Quantity': 'sum',
    'InvoiceNo': 'nunique', # number of transactions
    'InvoiceDate': ['min', 'max'] # first and last sales
}).reset_index()

# Flatten column names

product_performance.columns = ['StockCode', 'Description', 'Total_revenue', 'Total_Quantity', 'Num_Transactions', 'First_Sale_Date', 'Last_Sale_Date']
product_performance

,StockCode,Description,Total_revenue,Total_Quantity,Num_Transactions,First_Sale_Date,Last_Sale_Date
0,10002,INFLATABLE POLITICAL GLOBE,699.550,823,49,2010-12-01 08:45:00,2011-04-18 12:56:00
1,10080,GROOVY CACTUS INFLATABLE,114.410,291,21,2011-02-27 13:47:00,2011-11-21 17:04:00
2,10120,DOGGY RUBBER,40.530,193,29,2010-12-03 11:19:00,2011-12-04 13:15:00
3,10125,MINI FUNKY DESIGN TAPES,930.300,1226,61,2010-12-01 12:23:00,2011-12-09 10:13:00
4,10133,COLOURING PENCILS BROWN TUBE,1143.610,2384,122,2010-12-01 12:15:00,2011-09-07 10:07:00
...,...,...,...,...,...,...,...
3892,C2,CARRIAGE,6686.000,134,133,2010-12-01 14:05:00,2011-12-05 10:18:00
3893,DOT,DOTCOM POSTAGE,11906.360,16,16,2011-08-30 10:49:00,2011-12-05 17:17:00
3894,M,Manual,53779.930,7173,253,2010-12-01 15:35:00,2011-12-08 13:50:00
3895,PADS,PADS TO MATCH ALL CUSHIONS,0.003,3,3,2011-04-15 09:27:00,2011-09-25 14:58:00


##### Analyze sales velocity (units sold per time period)

In [49]:
# changing date format to 'YYYY.MM.DD'

product_performance['First_Sale_Date'] = pd.to_datetime(product_performance['First_Sale_Date'].dt.strftime('%Y.%m.%d'))
product_performance['Last_Sale_Date'] = pd.to_datetime(product_performance['Last_Sale_Date'].dt.strftime('%Y.%m.%d'))

# creating column called 'Days_activity' from the substraction of 'Last_Sale_Date' and 'First_Sale_Date'

product_performance['Days_activity'] = (pd.to_datetime(product_performance['Last_Sale_Date']) - pd.to_datetime(product_performance['First_Sale_Date'])).dt.days + 1
product_performance['Days_activity'] = product_performance['Days_activity'].astype(int)

# calculating sales velocity (Unit per day)

product_performance['Sales_velocity'] = (product_performance['Total_Quantity'] / product_performance['Days_activity']).round(2)
product_performance

,StockCode,Description,Total_revenue,Total_Quantity,Num_Transactions,First_Sale_Date,Last_Sale_Date,Days_activity,Sales_velocity,Movment_category
0,10002,INFLATABLE POLITICAL GLOBE,699.550,823,49,2010-12-01,2011-04-18,139,5.92,fast_moving
1,10080,GROOVY CACTUS INFLATABLE,114.410,291,21,2011-02-27,2011-11-21,268,1.09,medium_moving
2,10120,DOGGY RUBBER,40.530,193,29,2010-12-03,2011-12-04,367,0.53,slow_moving
3,10125,MINI FUNKY DESIGN TAPES,930.300,1226,61,2010-12-01,2011-12-09,374,3.28,medium_moving
4,10133,COLOURING PENCILS BROWN TUBE,1143.610,2384,122,2010-12-01,2011-09-07,281,8.48,fast_moving
...,...,...,...,...,...,...,...,...,...,...
3892,C2,CARRIAGE,6686.000,134,133,2010-12-01,2011-12-05,370,0.36,slow_moving
3893,DOT,DOTCOM POSTAGE,11906.360,16,16,2011-08-30,2011-12-05,98,0.16,slow_moving
3894,M,Manual,53779.930,7173,253,2010-12-01,2011-12-08,373,19.23,fast_moving
3895,PADS,PADS TO MATCH ALL CUSHIONS,0.003,3,3,2011-04-15,2011-09-25,164,0.02,slow_moving


##### Top/Bottom Product Analysis

In [6]:
# Top 10 products by Total Revenue ($)

top_10_revenue = product_performance.nlargest(10, 'Total_revenue')[['StockCode',	'Description',	'Total_revenue']]

print(" |Top 10 Products by Total Revenue: |")
top_10_revenue.reset_index()

 |Top 10 Products by Total Revenue: |


,index,StockCode,Description,Total_revenue
0,2529,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60
1,1245,22423,REGENCY CAKESTAND 3 TIER,142592.95
2,3576,85123A,WHITE HANGING HEART T-LIGHT HOLDER,100448.15
3,3569,85099B,JUMBO BAG RED RETROSPOT,85220.78
4,2027,23166,MEDIUM CERAMIC TOP STORAGE JAR,81416.73
5,3896,POST,POSTAGE,77803.96
6,2607,47566,PARTY BUNTING,68844.33
7,2810,84879,ASSORTED COLOUR BIRD ORNAMENT,56580.34
8,3894,M,Manual,53779.93
9,1933,23084,RABBIT NIGHT LIGHT,51346.20


In [7]:
# Bottom 10 products by Total Revenue

bottom_10_revenue = product_performance[product_performance['Total_revenue'] > 0].nsmallest(10, 'Total_revenue')[['StockCode',	'Description',	'Total_revenue']]

print(" Bottom 10 Products by Total Revenue:")
bottom_10_revenue.reset_index()

 Bottom 10 Products by Total Revenue:


,index,StockCode,Description,Total_revenue
0,3895,PADS,PADS TO MATCH ALL CUSHIONS,0.003
1,2711,84227,HEN HOUSE W CHICK IN NEST,0.420
2,2261,23366,SET 12 COLOURING PENCILS DOILEY,0.650
3,398,21268,VINTAGE BLUE TINSEL REEL,0.840
4,2944,90084,PINK CRYSTAL GUITAR PHONE CHARM,0.850
5,2958,90104,PURPLE FRANGIPANI HAIRCLIP,0.850
6,3315,84201C,HAPPY BIRTHDAY CARD TEDDY/CAKE,0.950
7,3317,84206B,CAT WITH SUNGLASSES BLANK CARD,0.950
8,2840,84990,60 GOLD AND SILVER FAIRY CAKE CASES,1.100
9,2267,23370,SET 36 COLOURING PENCILS DOILEY,1.250


##### Fast vs Slow moving products

In [48]:
product_performance['Movment_category'] = pd.qcut(product_performance['Sales_velocity'], 3, labels=['slow_moving', 'medium_moving', 'fast_moving'])
product_performance

,StockCode,Description,Total_revenue,Total_Quantity,Num_Transactions,First_Sale_Date,Last_Sale_Date,Days_activity,Sales_velocity,Movment_category
0,10002,INFLATABLE POLITICAL GLOBE,699.550,823,49,2010-12-01,2011-04-18,139,5.92,fast_moving
1,10080,GROOVY CACTUS INFLATABLE,114.410,291,21,2011-02-27,2011-11-21,268,1.09,medium_moving
2,10120,DOGGY RUBBER,40.530,193,29,2010-12-03,2011-12-04,367,0.53,slow_moving
3,10125,MINI FUNKY DESIGN TAPES,930.300,1226,61,2010-12-01,2011-12-09,374,3.28,medium_moving
4,10133,COLOURING PENCILS BROWN TUBE,1143.610,2384,122,2010-12-01,2011-09-07,281,8.48,fast_moving
...,...,...,...,...,...,...,...,...,...,...
3892,C2,CARRIAGE,6686.000,134,133,2010-12-01,2011-12-05,370,0.36,slow_moving
3893,DOT,DOTCOM POSTAGE,11906.360,16,16,2011-08-30,2011-12-05,98,0.16,slow_moving
3894,M,Manual,53779.930,7173,253,2010-12-01,2011-12-08,373,19.23,fast_moving
3895,PADS,PADS TO MATCH ALL CUSHIONS,0.003,3,3,2011-04-15,2011-09-25,164,0.02,slow_moving


##### Inventory Analysis:

##### Inventory Turnover Analysis

In [58]:
# Extarcting months from 'InvoiceDate' for turnover analysis

df['Year_month'] = df['InvoiceDate'].dt.to_period('M')

# Monthly sales by product category

monthly_sales = df.groupby(['Year_month', 'Description'])\
    .agg({
        'Quantity': 'sum',
        'Total_revenue': 'sum'
    }).reset_index()

monthly_sales

,Year_month,Description,Quantity,Total_revenue
0,2010-12,4 PURPLE FLOCK DINNER CANDLES,14,35.70
1,2010-12,OVAL WALL MIRROR DIAMANTE,9,89.55
2,2010-12,SET 2 TEA TOWELS I LOVE LONDON,257,758.15
3,2010-12,10 COLOUR SPACEBOY PEN,547,464.95
4,2010-12,12 COLOURED PARTY BALLOONS,48,31.20
...,...,...,...,...
30647,2011-12,ZINC T-LIGHT HOLDER STAR LARGE,85,81.88
30648,2011-12,ZINC T-LIGHT HOLDER STARS SMALL,120,100.40
30649,2011-12,ZINC WILLIE WINKIE CANDLE STICK,135,117.09
30650,2011-12,ZINC WIRE KITCHEN ORGANISER,16,63.20


In [47]:
# Monthly sales analysis for turnover

monthly_turnover = monthly_sales.groupby('Description')\
    .agg({
        'Quantity': ['mean', 'std']
    }).round(2).reset_index()
monthly_turnover.columns = ['Description', 'AvgMonthlySales', 'SalesStdDev']
monthly_turnover 

,Description,AvgMonthlySales,SalesStdDev
0,4 PURPLE FLOCK DINNER CANDLES,11.67,16.98
1,50'S CHRISTMAS GIFT BAG LARGE,377.00,394.45
2,DOLLY GIRL BEAKER,399.67,359.48
3,I LOVE LONDON MINI BACKPACK,90.00,63.57
4,I LOVE LONDON MINI RUCKSACK,1.00,NaN
...,...,...,...
3872,ZINC T-LIGHT HOLDER STARS SMALL,543.78,343.36
3873,ZINC TOP 2 DOOR WOODEN SHELF,1.67,1.63
3874,ZINC WILLIE WINKIE CANDLE STICK,200.54,165.83
3875,ZINC WIRE KITCHEN ORGANISER,5.00,6.20


In [46]:
# Coefficient of variation

monthly_turnover['cofficient_of_variation'] = (monthly_turnover['SalesStdDev']/monthly_turnover['AvgMonthlySales']).round(2)
monthly_turnover

,Description,AvgMonthlySales,SalesStdDev,cofficient_of_variation,stokeout_risk
0,4 PURPLE FLOCK DINNER CANDLES,11.67,16.98,1.46,High_risk
1,50'S CHRISTMAS GIFT BAG LARGE,377.00,394.45,1.05,High_risk
2,DOLLY GIRL BEAKER,399.67,359.48,0.90,meduim_risk
3,I LOVE LONDON MINI BACKPACK,90.00,63.57,0.71,meduim_risk
4,I LOVE LONDON MINI RUCKSACK,1.00,NaN,NaN,NaN
...,...,...,...,...,...
3872,ZINC T-LIGHT HOLDER STARS SMALL,543.78,343.36,0.63,low_risk
3873,ZINC TOP 2 DOOR WOODEN SHELF,1.67,1.63,0.98,High_risk
3874,ZINC WILLIE WINKIE CANDLE STICK,200.54,165.83,0.83,meduim_risk
3875,ZINC WIRE KITCHEN ORGANISER,5.00,6.20,1.24,High_risk


#####  Stockout risk indicator (high CV = unpredictable demand = higher stockout risk)

In [45]:
monthly_turnover['stokeout_risk'] = pd.qcut(monthly_turnover['cofficient_of_variation'], 3, labels=['low_risk', 'meduim_risk', 'High_risk'])
monthly_turnover

,Description,AvgMonthlySales,SalesStdDev,cofficient_of_variation,stokeout_risk
0,4 PURPLE FLOCK DINNER CANDLES,11.67,16.98,1.46,High_risk
1,50'S CHRISTMAS GIFT BAG LARGE,377.00,394.45,1.05,High_risk
2,DOLLY GIRL BEAKER,399.67,359.48,0.90,meduim_risk
3,I LOVE LONDON MINI BACKPACK,90.00,63.57,0.71,meduim_risk
4,I LOVE LONDON MINI RUCKSACK,1.00,NaN,NaN,NaN
...,...,...,...,...,...
3872,ZINC T-LIGHT HOLDER STARS SMALL,543.78,343.36,0.63,low_risk
3873,ZINC TOP 2 DOOR WOODEN SHELF,1.67,1.63,0.98,High_risk
3874,ZINC WILLIE WINKIE CANDLE STICK,200.54,165.83,0.83,meduim_risk
3875,ZINC WIRE KITCHEN ORGANISER,5.00,6.20,1.24,High_risk


##### Seasonal & Trend Analysis

In [44]:
# Seasonal Analysis

df['month'] = df['InvoiceDate'].dt.month
df['year'] = df['InvoiceDate'].dt.year

# Monthly sales trend for top category

monthly_trend = df.groupby(['year', 'month', 'Description'])\
    .agg({
        'Quantity': 'sum',
        'Total_revenue': 'sum'
    }).reset_index()

monthly_trend

,year,month,Description,Quantity,Total_revenue
0,2010,12,4 PURPLE FLOCK DINNER CANDLES,14,35.70
1,2010,12,OVAL WALL MIRROR DIAMANTE,9,89.55
2,2010,12,SET 2 TEA TOWELS I LOVE LONDON,257,758.15
3,2010,12,10 COLOUR SPACEBOY PEN,547,464.95
4,2010,12,12 COLOURED PARTY BALLOONS,48,31.20
...,...,...,...,...,...
30647,2011,12,ZINC T-LIGHT HOLDER STAR LARGE,85,81.88
30648,2011,12,ZINC T-LIGHT HOLDER STARS SMALL,120,100.40
30649,2011,12,ZINC WILLIE WINKIE CANDLE STICK,135,117.09
30650,2011,12,ZINC WIRE KITCHEN ORGANISER,16,63.20


In [60]:
# Identify seasonal patterns  

seasonal_products = monthly_trend.groupby('Description')\
    .agg({
        'Quantity': 'std'
    }).reset_index()
seasonal_products['Seasonality_score'] = pd.qcut(seasonal_products['Quantity'], 3, labels=['low', 'medium', 'high'])
seasonal_products

,Description,Quantity,Seasonality_score
0,4 PURPLE FLOCK DINNER CANDLES,16.977704,low
1,50'S CHRISTMAS GIFT BAG LARGE,394.449617,high
2,DOLLY GIRL BEAKER,359.477213,high
3,I LOVE LONDON MINI BACKPACK,63.566238,medium
4,I LOVE LONDON MINI RUCKSACK,NaN,NaN
...,...,...,...
3872,ZINC T-LIGHT HOLDER STARS SMALL,343.361609,high
3873,ZINC TOP 2 DOOR WOODEN SHELF,1.632993,low
3874,ZINC WILLIE WINKIE CANDLE STICK,165.832554,high
3875,ZINC WIRE KITCHEN ORGANISER,6.204837,low


In [66]:
# Declining products (last 3 months vs previous 3 months)

current_date = df['InvoiceDate'].max()
three_month_ago = current_date - pd.DateOffset(month=3)
six_month_ago = current_date - pd.DateOffset(month=6)

recent_sales = df[df['InvoiceDate'] > three_month_ago].groupby('Description')['Quantity'].sum()
previous_sales = df[(df['InvoiceDate'] > six_month_ago) & (df['InvoiceDate'] <= three_month_ago)].groupby('Description')['Quantity'].sum()

sales_comparision = pd.DataFrame({
    'recent_sales': recent_sales,
    'previous_sales': previous_sales
}).fillna(0)

sales_comparision['growth_rate'] = (sales_comparision['recent_sales'] - sales_comparision['previous_sales'])/sales_comparision['previous_sales'].replace(0, np.nan)

sales_comparision['Trend'] = np.where(sales_comparision['growth_rate'] < 0.2, 'Decline',
                                      np.where(sales_comparision['growth_rate'] > 0.2, 'growth', 'stable'))

sales_comparision

,recent_sales,previous_sales,growth_rate,Trend
Description,,,,
4 PURPLE FLOCK DINNER CANDLES,123,0.0,NaN,stable
50'S CHRISTMAS GIFT BAG LARGE,1885,0.0,NaN,stable
DOLLY GIRL BEAKER,2398,0.0,NaN,stable
I LOVE LONDON MINI BACKPACK,360,0.0,NaN,stable
I LOVE LONDON MINI RUCKSACK,1,0.0,NaN,stable
...,...,...,...,...
ZINC T-LIGHT HOLDER STARS SMALL,4894,0.0,NaN,stable
ZINC TOP 2 DOOR WOODEN SHELF,3,0.0,NaN,stable
ZINC WILLIE WINKIE CANDLE STICK,2175,0.0,NaN,stable
